# [실습 11] 한국어 ASR 강화학습 — WER 보상 기반 REINFORCE (Minimum Risk Training)

**목표** — 음성 도메인에서의 **강화학습 미세조정**을 직접 구현합니다.

[실습10] 에서 학습한 Whisper 한국어 ASR 모델은 토큰 단위 cross-entropy 로만 학습되어 있어
1) **시퀀스 단위 평가 지표(WER)** 와 학습 손실이 일치하지 않고
2) **exposure bias** — 학습 시 정답만 보다가, 추론 시 자기 출력을 보며 누적 오류가 폭증

이 두 문제를 해결하기 위해 **Minimum Risk Training (MRT)** — 본질적으로 시퀀스 레벨 **REINFORCE** — 으로 정책을 갱신합니다.

## 핵심 아이디어

현재 정책(Whisper)로 **K개의 가설**을 샘플링하고, 각 가설에 대해 **보상 = $1-\text{WER}$** 을 계산한 뒤, 정책 경사로 업데이트:

$$\mathcal{L}_{\text{MRT}} = -\sum_{k=1}^{K} \log \pi_\theta(y^{(k)} | x) \cdot \big(r^{(k)} - \bar{r}\big)$$

- $\bar{r}$ = 배치 내 가설들의 보상 평균(분산 감소를 위한 **baseline**)
- $y^{(k)}$ = $k$ 번째 샘플링된 토큰 시퀀스
- $r^{(k)} = 1 - \text{WER}(y^{(k)}, y^*)$

💡 RLHF 의 PPO 와 같은 정책경사 가족. 보상이 사람 선호도가 아니라 **자동 측정 가능한 WER** 일 뿐.

| 항목 | 값 |
|---|---|
| 출발 정책 | [실습10] 의 SFT Whisper (없으면 사전학습 Whisper-tiny) |
| 데이터 | `kresnik/zeroth_korean` |
| 보상 | $1 - \text{WER}$ |
| 알고리즘 | REINFORCE with mean baseline (≈ Minimum Risk Training) |
| 권장 환경 | Colab T4 |

## 0. 환경 준비

In [ ]:
!pip install -q --upgrade \
    transformers==4.44.2 datasets==2.21.0 accelerate==0.34.2 \
    librosa soundfile jiwer evaluate

In [ ]:
import os, math, torch
import torch.nn.functional as F
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

---
## 1. 정책(Whisper) 로드

[실습10] 의 산출물 `./whisper_ko_sft_final` 이 있으면 그걸, 없으면 `openai/whisper-tiny` 를 그대로 시작점으로 사용합니다.

In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

SFT_DIR = "./whisper_ko_sft_final"
if os.path.isdir(SFT_DIR):
    print(f"[OK] SFT 모델 사용: {SFT_DIR}")
    processor = WhisperProcessor.from_pretrained(SFT_DIR, language="korean", task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained(SFT_DIR).to(device)
else:
    print("[INFO] SFT 산출물이 없어 사전학습 Whisper-tiny 로 시작합니다.")
    processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="korean", task="transcribe")
    model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny").to(device)

model.generation_config.language = "korean"
model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

---
## 2. 데이터 준비

RL 미세조정은 SFT 보다 메모리/시간이 훨씬 많이 들기 때문에 더 작은 부분만 사용합니다.

In [ ]:
from datasets import load_dataset, Audio

rl_train = load_dataset("kresnik/zeroth_korean", split="train[:500]")
rl_eval  = load_dataset("kresnik/zeroth_korean", split="test[:100]")
rl_train = rl_train.cast_column("audio", Audio(sampling_rate=16000))
rl_eval  = rl_eval.cast_column("audio",  Audio(sampling_rate=16000))
print("RL train:", len(rl_train), "| eval:", len(rl_eval))

### 학습 전 WER (baseline)

In [ ]:
import evaluate
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

@torch.no_grad()
def greedy_transcribe(audio_arrays):
    inp = processor.feature_extractor(audio_arrays, sampling_rate=16000, return_tensors="pt").to(device)
    out = model.generate(input_features=inp.input_features, max_new_tokens=128, num_beams=1, do_sample=False)
    return processor.batch_decode(out, skip_special_tokens=True)

def eval_wer(ds, batch_size=8, n=None):
    n = n or len(ds)
    preds, refs = [], []
    for i in range(0, n, batch_size):
        chunk = [ds[j] for j in range(i, min(i + batch_size, n))]
        arrs = [s["audio"]["array"] for s in chunk]
        preds.extend(greedy_transcribe(arrs))
        refs.extend([s["text"] for s in chunk])
    return (wer_metric.compute(predictions=preds, references=refs),
            cer_metric.compute(predictions=preds, references=refs),
            preds, refs)

wer0, cer0, p0, r0 = eval_wer(rl_eval, n=80)
print(f"[RL 시작 전] WER = {wer0*100:.2f}%  | CER = {cer0*100:.2f}%")
for i in range(3):
    print(f" GT  : {r0[i]}")
    print(f" PRED: {p0[i]}")
    print()

---
## 3. REINFORCE 핵심 — 샘플 가설 + 보상 + 정책경사

### 3-1. 가설 K개 샘플링 + log π 계산

Whisper 의 `generate(do_sample=True, num_return_sequences=K, return_dict_in_generate=True, output_scores=True)` 로 가설을 뽑고,
`compute_transition_scores` 로 각 토큰의 log 확률을 가져옵니다.

In [ ]:
from jiwer import wer as jwer

def sample_hypotheses(audio_arrays, K=4, temperature=1.0, max_new_tokens=64):
    """K개의 가설과, 각 토큰의 log π(a_t|s_t) 를 함께 반환."""
    inp = processor.feature_extractor(audio_arrays, sampling_rate=16000, return_tensors="pt").to(device)
    B = inp.input_features.size(0)

    out = model.generate(
        input_features=inp.input_features,
        do_sample=True,
        num_return_sequences=K,
        temperature=temperature,
        max_new_tokens=max_new_tokens,
        return_dict_in_generate=True,
        output_scores=True,
    )
    # 생성된 토큰별 log 확률 합 (B*K, T)
    trans = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
    # padding(또는 EOS 이후) 토큰 마스킹
    eos = processor.tokenizer.eos_token_id
    seqs = out.sequences
    gen_len = seqs.size(1) - (inp.input_features.size(-1) * 0)  # decoder-only 시작
    # transition score 의 길이만큼만 사용
    mask = torch.ones_like(trans, dtype=torch.bool)
    # EOS 이후 토큰은 마스킹
    eos_pos = (out.sequences[:, -trans.size(1):] == eos)
    cum_eos = torch.cumsum(eos_pos.int(), dim=1) > 0
    # EOS 자신은 포함, 그 다음부터 마스킹
    after_eos = torch.cat([torch.zeros_like(cum_eos[:, :1]), cum_eos[:, :-1]], dim=1)
    mask = ~after_eos

    log_probs = (trans * mask).sum(dim=1)            # (B*K,) 시퀀스별 log π
    texts = processor.batch_decode(out.sequences, skip_special_tokens=True)
    return texts, log_probs.view(B, K), out.sequences.view(B, K, -1)

# 잘 돌아가는지 한 번 확인
sample_batch = [rl_train[i] for i in range(2)]
txts, lps, seqs = sample_hypotheses([s["audio"]["array"] for s in sample_batch], K=3, temperature=1.0)
for b, s in enumerate(sample_batch):
    print(f"GT: {s['text']}")
    for k in range(3):
        print(f"  K{k}  logπ={lps[b,k].item():.2f}  → {txts[b*3 + k]}")
    print()

### 3-2. 보상 함수: $r = 1 - \text{WER}$

In [ ]:
def reward_fn(hypothesis: str, reference: str) -> float:
    # 빈 가설/참조 보호
    if not reference.strip():
        return 0.0
    try:
        w = jwer(reference, hypothesis or " ")
    except Exception:
        return 0.0
    # 1 - WER, [0, 1] 로 클리핑
    return float(max(0.0, min(1.0, 1.0 - w)))

# 직관 체크
print(reward_fn("나는 학교에 갑니다", "나는 학교에 갑니다"))          # 1.0
print(reward_fn("나는 학교에 갔다",    "나는 학교에 갑니다"))         # 부분 일치
print(reward_fn("전혀 다른 문장",     "나는 학교에 갑니다"))          # 낮음

### 3-3. REINFORCE 손실

동일 입력에 대해 K개 샘플 → 보상 평균을 빼서 baseline 으로 사용 (분산 감소).

$$\mathcal{L} = -\frac{1}{B \cdot K} \sum_{b,k} \log \pi_\theta(y^{(b,k)} | x_b) \cdot \big(r^{(b,k)} - \bar{r}_b\big)$$

In [ ]:
def reinforce_loss(log_probs_BK: torch.Tensor, rewards_BK: torch.Tensor) -> torch.Tensor:
    # 배치 내 baseline: 각 입력별 K 평균 보상
    baseline = rewards_BK.mean(dim=1, keepdim=True)
    advantage = rewards_BK - baseline
    # 정책경사: -E[ logπ * advantage ]
    loss = -(log_probs_BK * advantage).mean()
    return loss

---
## 4. RL 학습 루프

수동으로 매 스텝마다 (샘플 → 보상 → 손실 → 역전파) 를 반복합니다.  
Whisper 전체를 학습하면 메모리가 빠듯하므로, **인코더는 freeze 하고 디코더만 업데이트** 합니다.

In [ ]:
# 인코더 freeze — 음성 표현은 그대로 두고 토큰 생성 정책만 갱신
for p in model.model.encoder.parameters():
    p.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"학습 파라미터: {trainable/1e6:.2f}M / {total/1e6:.2f}M")

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-6)

In [ ]:
# 학습 하이퍼파라미터
NUM_STEPS = 80           # 데모용 — 본격 RL 은 수백~수천 스텝
BATCH_SIZE = 2           # 입력 발화 개수
K = 4                    # 발화당 가설 샘플 개수
TEMPERATURE = 1.0
LOG_EVERY = 5

import random
random.seed(0)
history = {"step": [], "loss": [], "reward": [], "reward_std": []}

model.train()
for step in range(1, NUM_STEPS + 1):
    # 1) 미니배치 추출
    idx = random.sample(range(len(rl_train)), BATCH_SIZE)
    batch = [rl_train[i] for i in idx]
    audios = [s["audio"]["array"] for s in batch]
    refs   = [s["text"]            for s in batch]

    # 2) 가설 K개 샘플링 + log π
    hyps, log_probs_BK, _ = sample_hypotheses(audios, K=K, temperature=TEMPERATURE)

    # 3) 보상 계산
    rewards = torch.zeros(BATCH_SIZE, K, device=device)
    for b in range(BATCH_SIZE):
        for k in range(K):
            rewards[b, k] = reward_fn(hyps[b * K + k], refs[b])

    # 4) REINFORCE loss
    loss = reinforce_loss(log_probs_BK, rewards)

    # 5) 역전파
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    history["step"].append(step)
    history["loss"].append(loss.item())
    history["reward"].append(rewards.mean().item())
    history["reward_std"].append(rewards.std().item())

    if step == 1 or step % LOG_EVERY == 0:
        print(f"step {step:>3} | loss {loss.item():+.4f} | "
              f"reward μ={rewards.mean():.3f} σ={rewards.std():.3f} | "
              f"best={rewards.max():.3f}")

print("\nRL 학습 종료")

---
## 5. 학습 곡선 시각화

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))

axes[0].plot(history["step"], history["loss"], marker="o", linewidth=1)
axes[0].set_title("REINFORCE Loss"); axes[0].grid(alpha=0.3)
axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")

axes[1].plot(history["step"], history["reward"], marker="o", linewidth=1, label="mean reward")
axes[1].fill_between(history["step"],
                     [m - s for m, s in zip(history["reward"], history["reward_std"])],
                     [m + s for m, s in zip(history["reward"], history["reward_std"])],
                     alpha=0.2, label="±1σ")
axes[1].set_title("Sample Reward (1 - WER)"); axes[1].grid(alpha=0.3); axes[1].legend()
axes[1].set_xlabel("step"); axes[1].set_ylabel("reward")

plt.tight_layout(); plt.show()

---
## 6. 학습 전/후 평가

In [ ]:
model.eval()
wer1, cer1, p1, r1 = eval_wer(rl_eval, n=80)
print(f"[BEFORE RL] WER = {wer0*100:.2f}%  | CER = {cer0*100:.2f}%")
print(f"[AFTER  RL] WER = {wer1*100:.2f}%  | CER = {cer1*100:.2f}%")
print(f"           \u0394WER = {(wer1-wer0)*100:+.2f}%p")
print("-" * 60)
for i in range(5):
    print(f"GT    : {r1[i]}")
    print(f"BEFORE: {p0[i]}")
    print(f"AFTER : {p1[i]}")
    print()

In [ ]:
# 모델 저장 (옵션)
model.save_pretrained("./whisper_ko_rl")
processor.save_pretrained("./whisper_ko_rl")
print("저장 경로: ./whisper_ko_rl")

---
## 7. 음성 도메인 RL — 정리

### 오늘 한 일
1. Whisper 정책으로 한 발화당 K개 가설을 **샘플링**
2. 각 가설에 대해 **1−WER** 보상 계산
3. 동일 입력의 K개 평균을 baseline 으로 사용 → 분산 감소
4. REINFORCE 로 디코더 파라미터 업데이트

### 보너스: 다른 보상도 가능
보상 함수만 바꾸면 다양한 음성 RL 가능:
- **CER 기반** : 한국어처럼 형태소 변화가 많을 때 더 안정적
- **BLEU/ROUGE** : 자유 발화 인식
- **MOS 모델 출력** : TTS 의 자연스러움 향상 (생성 음성 품질)
- **사람 선호도** : 진짜 RLHF for Speech

### 주의점
- ❗ **lr 은 SFT 보다 10~100× 작게** (1e-6 권장). 크면 정책이 무너집니다.
- ❗ 보상이 모두 같으면 (`advantage=0`) 그래디언트가 사라짐 → temperature 를 약간 높여 다양성 확보.
- ❗ Reward Hacking 주의 — "빈 출력 = WER 100%" 같은 함정 보상이 있으면 정책이 그쪽으로 무너질 수 있음.

### 응용 과제
- 🔧 **A.** baseline 을 K-평균이 아니라 학습 가능한 value-head로 바꿔 Actor-Critic 으로 확장
- 🔧 **B.** PPO 처럼 ratio clipping 추가 (`trl.PPOTrainer` 의 핵심)
- 🔧 **C.** 보상 = 0.7·(1−WER) + 0.3·(1−CER) 처럼 다중 보상 결합
- 🔧 **D.** 사람 평가자가 두 인식 결과를 비교한 한국어 선호도 데이터를 만들어 **Speech-DPO** 구현